# 📄 README Source Filter

This notebook filters JSON entries to find those with README.md sources.

**Features:**
- Load JSON data from specified path
- Filter entries with README.md sources
- Export filtered results
- Display statistics and sample entries

## 1) 📦 Imports & Setup

In [1]:
import json
import os
import re
from typing import List, Dict, Any
from urllib.parse import urlparse
import pandas as pd
from collections import Counter

print("Setup complete!")

Setup complete!


## 2) ⚙️ Configuration

In [2]:
# Configuration - Update this path to your JSON file
JSON_PATH = "test.json"  # Update this path

# Output configuration
OUTPUT_DIR = "./readme_filtered_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# README patterns to match
README_PATTERNS = [
    r'.*[Rr][Ee][Aa][Dd][Mm][Ee]\.[Mm][Dd]$',
    r'.*[Rr][Ee][Aa][Dd][Mm][Ee]$',
    r'.*readme\.[Mm][Dd]$',
    r'.*README\.[Mm][Dd]$'
]

print(f"JSON Path: {JSON_PATH}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"README Patterns: {len(README_PATTERNS)} patterns configured")

JSON Path: test.json
Output Directory: ./readme_filtered_output
README Patterns: 4 patterns configured


## 3) 📂 Load JSON Data

In [3]:
def load_json_data(file_path: str) -> List[Dict[str, Any]]:
    """Load JSON data from file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Handle different JSON structures
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            # If it's a dict, try to find the main data array
            for key in ['data', 'items', 'entries', 'results']:
                if key in data and isinstance(data[key], list):
                    return data[key]
            # If no standard key found, return as single item list
            return [data]
        else:
            print(f"Warning: Unexpected data type: {type(data)}")
            return []
    
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        return []
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return []

# Load the data
print(f"Loading data from: {JSON_PATH}")
data = load_json_data(JSON_PATH)

if data:
    print(f"✅ Loaded {len(data)} entries")
    print(f"Sample entry keys: {list(data[0].keys()) if data else 'No data'}")
else:
    print("❌ No data loaded. Please check the JSON_PATH configuration.")

Loading data from: test.json
✅ Loaded 1 entries
Sample entry keys: ['somef_provenance', 'code_repository', 'owner', 'date_created', 'date_updated', 'license', 'description', 'name', 'full_name', 'issue_tracker', 'forks_url', 'stargazers_count', 'keywords', 'forks_count', 'homepage', 'download_url', 'programming_languages', 'releases', 'code_of_conduct', 'has_build_file', 'has_package_file', 'package_id', 'version', 'requirements', 'documentation', 'related_documentation', 'authors', 'readme_url', 'contributing_guidelines', 'executable_example', 'installation', 'has_script_file', 'continuous_integration', 'type', 'usage', 'support', 'full_title', 'support_channels', 'package_distribution', 'images']


## 4) 🔍 Analyze Data Structure

In [4]:
def analyze_data_structure(data: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Analyze the structure of the data to find source fields"""
    if not data:
        return {}
    
    # Find all possible keys that might contain source information
    all_keys = set()
    source_like_keys = []
    
    for entry in data[:10]:  # Sample first 10 entries
        if isinstance(entry, dict):
            all_keys.update(entry.keys())
    
    # Look for keys that might contain source/URL information
    for key in all_keys:
        key_lower = key.lower()
        if any(term in key_lower for term in ['source', 'url', 'link', 'path', 'file', 'origin']):
            source_like_keys.append(key)
    
    return {
        'total_entries': len(data),
        'all_keys': sorted(all_keys),
        'source_like_keys': source_like_keys,
        'sample_entry': data[0] if data else None
    }

if data:
    analysis = analyze_data_structure(data)
    
    print(f"📊 Data Structure Analysis:")
    print(f"  Total entries: {analysis['total_entries']}")
    print(f"  All keys found: {analysis['all_keys']}")
    print(f"  Source-like keys: {analysis['source_like_keys']}")
    
    if analysis['sample_entry']:
        print(f"\n📋 Sample entry:")
        for key, value in list(analysis['sample_entry'].items())[:5]:
            print(f"  {key}: {str(value)[:100]}{'...' if len(str(value)) > 100 else ''}")
else:
    print("No data to analyze")

📊 Data Structure Analysis:
  Total entries: 1
  All keys found: ['authors', 'code_of_conduct', 'code_repository', 'continuous_integration', 'contributing_guidelines', 'date_created', 'date_updated', 'description', 'documentation', 'download_url', 'executable_example', 'forks_count', 'forks_url', 'full_name', 'full_title', 'has_build_file', 'has_package_file', 'has_script_file', 'homepage', 'images', 'installation', 'issue_tracker', 'keywords', 'license', 'name', 'owner', 'package_distribution', 'package_id', 'programming_languages', 'readme_url', 'related_documentation', 'releases', 'requirements', 'somef_provenance', 'stargazers_count', 'support', 'support_channels', 'type', 'usage', 'version']
  Source-like keys: ['download_url', 'readme_url', 'forks_url', 'has_script_file', 'has_package_file', 'has_build_file']

📋 Sample entry:
  somef_provenance: {'somef_version': '0.9.12', 'somef_schema_version': '1.0.0', 'date': '2025-10-25 20:03:10'}
  code_repository: [{'result': {'value': 'htt

## 5) 🔎 Filter README Sources

In [5]:
def is_readme_source(value: str) -> bool:
    """Check if a value contains a README source"""
    if not isinstance(value, str):
        return False
    
    # Check against all README patterns
    for pattern in README_PATTERNS:
        if re.search(pattern, value):
            return True
    return False

def find_readme_entries(data: List[Dict[str, Any]], source_keys: List[str] = None) -> List[Dict[str, Any]]:
    """Find entries with README sources"""
    readme_entries = []
    
    # If no source keys specified, search all string fields
    if not source_keys:
        source_keys = ['source', 'url', 'link', 'path', 'file_path', 'origin', 'location']
    
    for entry in data:
        if not isinstance(entry, dict):
            continue
        
        found_readme = False
        readme_sources = []
        
        # Check specified source keys
        for key in source_keys:
            if key in entry:
                value = entry[key]
                if is_readme_source(str(value)):
                    found_readme = True
                    readme_sources.append({key: value})
        
        # If not found in specific keys, search all string values
        if not found_readme:
            for key, value in entry.items():
                if isinstance(value, str) and is_readme_source(value):
                    found_readme = True
                    readme_sources.append({key: value})
                    break
        
        if found_readme:
            # Add metadata about found sources
            entry_copy = entry.copy()
            entry_copy['_readme_sources'] = readme_sources
            readme_entries.append(entry_copy)
    
    return readme_entries

# Filter entries with README sources
if data:
    # Use the source-like keys found in analysis, or common ones
    source_keys = analysis.get('source_like_keys', []) if 'analysis' in locals() else []
    
    print(f"🔍 Filtering entries with README sources...")
    print(f"Looking in keys: {source_keys if source_keys else 'all string fields'}")
    
    readme_entries = find_readme_entries(data, source_keys)
    
    print(f"\n✅ Found {len(readme_entries)} entries with README sources")
    print(f"📊 Percentage: {len(readme_entries)/len(data)*100:.1f}% of total entries")
    
    if readme_entries:
        # Show sample README sources found
        print(f"\n📋 Sample README sources:")
        for i, entry in enumerate(readme_entries[:5]):
            sources = entry.get('_readme_sources', [])
            print(f"  {i+1}. {sources}")
else:
    readme_entries = []
    print("No data to filter")

🔍 Filtering entries with README sources...
Looking in keys: ['download_url', 'readme_url', 'forks_url', 'has_script_file', 'has_package_file', 'has_build_file']

✅ Found 0 entries with README sources
📊 Percentage: 0.0% of total entries


## 6) 📈 Statistics & Analysis

In [6]:
if readme_entries:
    print(f"📊 README Entries Statistics:")
    print(f"  Total README entries: {len(readme_entries)}")
    
    # Analyze source field distribution
    source_field_counts = Counter()
    all_readme_urls = []
    
    for entry in readme_entries:
        sources = entry.get('_readme_sources', [])
        for source_info in sources:
            for field, url in source_info.items():
                source_field_counts[field] += 1
                all_readme_urls.append(url)
    
    print(f"\n📋 Source fields containing README links:")
    for field, count in source_field_counts.most_common():
        print(f"  {field}: {count} entries")
    
    # Analyze URL patterns
    domains = Counter()
    for url in all_readme_urls:
        try:
            parsed = urlparse(str(url))
            if parsed.netloc:
                domains[parsed.netloc] += 1
        except:
            continue
    
    if domains:
        print(f"\n🌐 Top domains with README links:")
        for domain, count in domains.most_common(10):
            print(f"  {domain}: {count} links")
    
    # Show sample entries
    print(f"\n📄 Sample README entries:")
    for i, entry in enumerate(readme_entries[:3]):
        print(f"\n--- Entry {i+1} ---")
        # Show key fields (excluding the metadata we added)
        for key, value in entry.items():
            if key != '_readme_sources':
                print(f"  {key}: {str(value)[:150]}{'...' if len(str(value)) > 150 else ''}")
        
        # Show README sources
        sources = entry.get('_readme_sources', [])
        print(f"  README sources: {sources}")
else:
    print("No README entries found to analyze")

No README entries found to analyze


## 7) 💾 Export Results

In [7]:
if readme_entries:
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Export filtered entries as JSON
    json_output_path = os.path.join(OUTPUT_DIR, f"readme_entries_{timestamp}.json")
    with open(json_output_path, 'w', encoding='utf-8') as f:
        json.dump(readme_entries, f, indent=2, ensure_ascii=False)
    
    # Export summary statistics
    summary = {
        "timestamp": timestamp,
        "total_original_entries": len(data) if data else 0,
        "readme_entries_found": len(readme_entries),
        "percentage": len(readme_entries)/len(data)*100 if data else 0,
        "source_field_distribution": dict(source_field_counts) if 'source_field_counts' in locals() else {},
        "top_domains": dict(domains.most_common(10)) if 'domains' in locals() else {},
        "readme_patterns_used": README_PATTERNS
    }
    
    summary_path = os.path.join(OUTPUT_DIR, f"readme_filter_summary_{timestamp}.json")
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    # Export as CSV for easy viewing
    if readme_entries:
        # Flatten the data for CSV
        csv_data = []
        for entry in readme_entries:
            row = entry.copy()
            # Convert README sources to string
            row['readme_sources'] = str(row.pop('_readme_sources', []))
            csv_data.append(row)
        
        df = pd.DataFrame(csv_data)
        csv_path = os.path.join(OUTPUT_DIR, f"readme_entries_{timestamp}.csv")
        df.to_csv(csv_path, index=False, encoding='utf-8')
    
    # Export just the README URLs
    if 'all_readme_urls' in locals():
        urls_path = os.path.join(OUTPUT_DIR, f"readme_urls_{timestamp}.txt")
        with open(urls_path, 'w', encoding='utf-8') as f:
            for url in sorted(set(all_readme_urls)):
                f.write(f"{url}\n")
    
    print(f"✅ Export completed!")
    print(f"📁 Output directory: {OUTPUT_DIR}")
    print(f"📄 Files created:")
    print(f"  - readme_entries_{timestamp}.json ({len(readme_entries)} entries)")
    print(f"  - readme_entries_{timestamp}.csv (tabular format)")
    print(f"  - readme_filter_summary_{timestamp}.json (statistics)")
    if 'all_readme_urls' in locals():
        print(f"  - readme_urls_{timestamp}.txt ({len(set(all_readme_urls))} unique URLs)")
else:
    print("❌ No README entries to export")

❌ No README entries to export


## 8) 📋 Summary

In [8]:
print("\n" + "="*60)
print("📄 README SOURCE FILTER - SUMMARY")
print("="*60)

if data:
    print(f"📊 Input: {len(data)} total entries")
    print(f"✅ Output: {len(readme_entries)} README entries found")
    print(f"📈 Success rate: {len(readme_entries)/len(data)*100:.1f}%")
    
    if readme_entries:
        print(f"\n🔍 README patterns matched:")
        for pattern in README_PATTERNS:
            print(f"  - {pattern}")
        
        if 'source_field_counts' in locals():
            print(f"\n📋 Fields containing README links:")
            for field, count in source_field_counts.most_common(5):
                print(f"  - {field}: {count} entries")
        
        print(f"\n💾 Results exported to: {OUTPUT_DIR}")
    else:
        print(f"\n❌ No README sources found")
        print(f"💡 Try checking:")
        print(f"  - JSON file structure")
        print(f"  - Field names containing source information")
        print(f"  - README URL patterns in your data")
else:
    print(f"❌ No data loaded from: {JSON_PATH}")
    print(f"💡 Please update the JSON_PATH configuration")

print("="*60)


📄 README SOURCE FILTER - SUMMARY
📊 Input: 1 total entries
✅ Output: 0 README entries found
📈 Success rate: 0.0%

❌ No README sources found
💡 Try checking:
  - JSON file structure
  - Field names containing source information
  - README URL patterns in your data
